In [5]:
import cv2
import numpy as np
import rawpy
import os

from tkinter import Tk
from tkinter.filedialog import askopenfilename

In [6]:
# --- KONFIGURASI ---
TARGET_GRAY_LEVEL_SRGB = 122.0  # Target kecerahan Gray Card (skala 0-255)
GAMMA = 2.2

def apply_gamma(image, gamma=GAMMA, max_val=65535):
    """Konversi Linear ke sRGB untuk display"""
    img_norm = image.astype(np.float32) / max_val
    img_gamma = np.power(img_norm, 1.0 / gamma)
    img_out = (img_gamma * 255).astype(np.uint8)
    return img_out

def load_image_data(filepath):
    """
    Load gambar. 
    - RAW: Dikembalikan sebagai 16-bit Linear RGB (0-65535).
    - JPG: Dikembalikan sebagai 8-bit BGR (0-255).
    """
    if not os.path.exists(filepath):
        print(f"[ERROR] File tidak ditemukan: {filepath}")
        return None, None, None

    ext = os.path.splitext(filepath)[1].lower()
    if ext in ['.dng', '.cr2', '.nef', '.arw']:
        print(f"[LOAD] Membaca RAW: {filepath}")
        with rawpy.imread(filepath) as raw:
            # Output 16-bit Linear
            rgb = raw.postprocess(gamma=(1,1), no_auto_bright=True, use_camera_wb=False, output_bps=16)
            image = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
            return image, "RAW", 65535
    else:
        print(f"[LOAD] Membaca Standard (JPG/PNG): {filepath}")
        image = cv2.imread(filepath)
        return image, "JPG", 255

def get_roi_mean(image, roi_coords):
    """Menghitung rata-rata warna BGR pada area tertentu"""
    x, y, w, h = roi_coords
    roi_data = image[y:y+h, x:x+w].astype(np.float32)
    return np.mean(roi_data, axis=(0,1)) # Return [Mean B, Mean G, Mean R]

def select_two_rois(image, title, is_raw=False):
    """
    Fungsi GUI untuk memilih DUA area:
    1. Gray Card (Untuk Kalibrasi)
    2. Konjungtiva/Target (Untuk Evaluasi Error)
    """
    # Buat preview yang ramah mata
    if is_raw:
        preview = apply_gamma(image, max_val=65535)
    else:
        preview = image.astype(np.uint8)
    
    # Resize agar muat di layar
    h, w = preview.shape[:2]
    scale = min(1000/w, 800/h)
    preview_small = cv2.resize(preview, (int(w*scale), int(h*scale)))
    
    # WINDOW 1: GRAY CARD
    win_gray = f"{title} - 1. PILIH GRAY CARD"
    cv2.namedWindow(win_gray, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win_gray, preview_small.shape[1], preview_small.shape[0])
    print(f"\n[{title}] pilih bagian GRAY CARD...")
    roi_gray = cv2.selectROI(win_gray, preview_small, showCrosshair=True)
    cv2.destroyWindow(win_gray)
    
    # WINDOW 2: KONJUNGTIVA
    win_target = f"{title} - 2. PILIH KONJUNGTIVA (TARGET)"
    cv2.namedWindow(win_target, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(win_target, preview_small.shape[1], preview_small.shape[0])
    print(f"[{title}] pilih bagian KONJUNGTIVA...")
    roi_target = cv2.selectROI(win_target, preview_small, showCrosshair=True)
    cv2.destroyWindow(win_target)
    
    # Kembalikan koordinat ke skala asli
    real_roi_gray = [int(c / scale) for c in roi_gray]
    real_roi_target = [int(c / scale) for c in roi_target]
    
    return real_roi_gray, real_roi_target

def calculate_correction_factors(mean_gray_bgr, max_val, is_raw):
    """Menghitung Gain WB dan Gain Exposure"""
    b, g, r = mean_gray_bgr
    
    # 1. White Balance Gain (Samakan ke Green)
    wb_gain_r = g / r if r > 0 else 1.0
    wb_gain_b = g / b if b > 0 else 1.0
    
    # 2. Exposure Gain (Targetkan ke nilai Gray standar)
    if is_raw:
        # Konversi target sRGB 122 ke Linear 16-bit
        target_norm = (TARGET_GRAY_LEVEL_SRGB / 255.0) ** GAMMA
        target_val = target_norm * 65535.0
    else:
        # JPG diasumsikan sudah Gamma, target langsung 122
        target_val = TARGET_GRAY_LEVEL_SRGB
        
    exp_gain = target_val / g if g > 0 else 1.0
    
    # Gabung
    total_r = wb_gain_r * exp_gain
    total_g = 1.0 * exp_gain
    total_b = wb_gain_b * exp_gain
    
    return (total_b, total_g, total_r)

def apply_correction(image, gains, max_val):
    """Menerapkan gain ke gambar"""
    gain_b, gain_g, gain_r = gains
    
    img_float = image.astype(np.float32)
    img_float[:,:,0] *= gain_b
    img_float[:,:,1] *= gain_g
    img_float[:,:,2] *= gain_r
    
    return np.clip(img_float, 0, max_val)

def calculate_percent_error(color1, color2):
    """Menghitung persentase perbedaan rata-rata RGB"""
    # Euclidean distance antar dua warna
    diff = np.linalg.norm(color1 - color2)
    # Rata-rata magnitud warna (untuk normalisasi persentase)
    avg_mag = (np.linalg.norm(color1) + np.linalg.norm(color2)) / 2.0
    
    if avg_mag == 0: return 0.0
    return (diff / avg_mag) * 100.0

def get_crop_visual(img, roi, is_raw):
    """Fungsi bantu untuk mengambil potongan gambar (crop)"""
    x, y, w, h = roi
    crop = img[y:y+h, x:x+w]
    if is_raw: crop = apply_gamma(crop, max_val=65535)
    else: crop = crop.astype(np.uint8)
    return cv2.resize(crop, (300, 300))

def show_full_comparison(img_a, img_b, title, is_raw_a, is_raw_b):
    """Menampilkan gambar utuh side-by-side"""
    # Normalisasi visual
    view_a = apply_gamma(img_a, max_val=65535) if is_raw_a else img_a.astype(np.uint8)
    view_b = apply_gamma(img_b, max_val=65535) if is_raw_b else img_b.astype(np.uint8)
    
    # Resize agar tinggi sama (max 600px)
    target_h = 600
    
    h_a, w_a = view_a.shape[:2]
    scale_a = target_h / h_a
    view_a_small = cv2.resize(view_a, (int(w_a * scale_a), target_h))
    
    h_b, w_b = view_b.shape[:2]
    scale_b = target_h / h_b
    view_b_small = cv2.resize(view_b, (int(w_b * scale_b), target_h))
    
    combined = np.hstack((view_a_small, view_b_small))
    
    # Tambahkan label teks dinamis
    label_status = "Original" if "SEBELUM" in title else "Corrected"
    cv2.putText(combined, f"hp najla ({label_status})", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    cv2.putText(combined, f"hpmu ({label_status})", (view_a_small.shape[1] + 10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    cv2.imshow(title, combined)

def process_comparison(path_a, path_b):
    # 1. Load Gambar
    img_a, type_a, max_a = load_image_data(path_a)
    img_b, type_b, max_b = load_image_data(path_b)
    
    if img_a is None or img_b is None: return

    print(f"\nMode Perbandingan: {type_a} vs {type_b}")
    
    # 2. Seleksi ROI (Gray Card & Konjungtiva)
    roi_gray_a, roi_target_a = select_two_rois(img_a, "hp najla", type_a == "RAW")
    roi_gray_b, roi_target_b = select_two_rois(img_b, "hpmu", type_b == "RAW")
    
    # --- ANALISIS BEFORE ---
    raw_gray_a = get_roi_mean(img_a, roi_gray_a)
    raw_gray_b = get_roi_mean(img_b, roi_gray_b)
    raw_target_a = get_roi_mean(img_a, roi_target_a)
    raw_target_b = get_roi_mean(img_b, roi_target_b)
    
    error_gray_before = calculate_percent_error(raw_gray_a, raw_gray_b)
    error_target_before = calculate_percent_error(raw_target_a, raw_target_b)
    
    # --- VISUALISASI SEBELUM (BEFORE) ---
    print("\n[VISUALISASI] SEBELUM koreksi...")
    
    # 1. Full Image Before
    show_full_comparison(img_a, img_b, "1. SEBELUM KOREKSI", type_a == "RAW", type_b == "RAW")
    
    # 2. Crop Konjungtiva Before
    vis_a_before = get_crop_visual(img_a, roi_target_a, type_a == "RAW")
    vis_b_before = get_crop_visual(img_b, roi_target_b, type_b == "RAW")
    combined_crop_before = np.hstack((vis_a_before, vis_b_before))
    
    cv2.putText(combined_crop_before, f"Err: {error_target_before:.1f}%", (10, 280), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    cv2.imshow("2. BAGIAN Konjungtiva", combined_crop_before)
    
    # --- PROSES KOREKSI ---
    gains_a = calculate_correction_factors(raw_gray_a, max_a, type_a == "RAW")
    gains_b = calculate_correction_factors(raw_gray_b, max_b, type_b == "RAW")
    
    img_a_corr = apply_correction(img_a, gains_a, max_a)
    img_b_corr = apply_correction(img_b, gains_b, max_b)
    
    # --- ANALISIS AFTER ---
    # Kita gunakan koordinat ROI yang sama
    corr_gray_a = get_roi_mean(img_a_corr, roi_gray_a)
    corr_gray_b = get_roi_mean(img_b_corr, roi_gray_b)
    corr_target_a = get_roi_mean(img_a_corr, roi_target_a)
    corr_target_b = get_roi_mean(img_b_corr, roi_target_b)
    
    error_gray_after = calculate_percent_error(corr_gray_a, corr_gray_b)
    error_target_after = calculate_percent_error(corr_target_a, corr_target_b)
    
    # --- LAPORAN DETAIL ---
    print("\n" + "="*60)
    print(f"HASIL EVALUASI: RAW 1 vs RAW 2")
    print("="*60)
    
    print(f"\n--- 1. ANALISIS GRAY CARD ---")
    print(f"pembuktian kalibrasi pencahayaan berhasil.")
    print(f"Error Gray Card Sebelum : {error_gray_before:.5f} %")
    print(f"Error Gray Card Sesudah : {error_gray_after:.5f} %")
    print(f"   > RGB HP A (Corrected): {corr_gray_a.astype(int)}")
    print(f"   > RGB HP B (Corrected): {corr_gray_b.astype(int)}")
    
    print(f"\n--- 2. ANALISIS KONJUNGTIVA ---")
    print(f"pembuktian sbeerapa error warna konjungtiva.")
    print(f"Error Konjungtiva Sebelum: {error_target_before:.5f} %")
    print(f"Error Konjungtiva Sesudah: {error_target_after:.5f} %")
    print(f"   > RGB HP A (Corrected): {corr_target_a.astype(int)}")
    print(f"   > RGB HP B (Corrected): {corr_target_b.astype(int)}")
    
    print("-" * 60)
    if error_gray_after < 2.0:
        print("Gray Card konsisten.")
        if error_target_after > 5.0:
             print("Device-Specific Bias keknya sensor kameranya")
    else:
        print("Gagal")

    # --- VISUALISASI SESUDAH (AFTER) ---
    # 3. Tampilkan Gambar Utuh (Full View)
    show_full_comparison(img_a_corr, img_b_corr, "3. HASIL COLOR CORRECTION (FULL)", type_a == "RAW", type_b == "RAW")

    # 4. Tampilkan Potongan Konjungtiva
    vis_a = get_crop_visual(img_a_corr, roi_target_a, type_a == "RAW")
    vis_b = get_crop_visual(img_b_corr, roi_target_b, type_b == "RAW")
    combined_crop = np.hstack((vis_a, vis_b))
    
    # Tambahkan teks ke visualisasi crop
    cv2.putText(combined_crop, f"Err: {error_target_after:.1f}%", (10, 280), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 2)
    cv2.imshow("4. Konjungtiva (SESUDAH)", combined_crop)
    
    #print("\nTekan sembarang tombol di jendela gambar untuk menutup...")
    cv2.waitKey(0)
    cv2.destroyAllWindows()

if __name__ == "__main__":
    # --- GANTI PATH DI SINI SESUAI RENCANA EVALUASI ANDA ---
    
    Tk().withdraw()  # Hide main window

    file_hp_A_raw = askopenfilename(title="Pilih Raw Image dari HP A")
    file_hp_B_raw = askopenfilename(title="Pilih Raw Image dari HP B")

    # Contoh penggunaan (Pastikan path file benar)
    #file_hp_A_raw = r"D:\FILE ITS\SEMESTER 7\Pra-TA\Keperluan Pra-Ta\Preliminary Pra-TA\Trying RawPy Library\buat prelim\hp najla\grey card1.dng" 
    #file_hp_B_raw = r"D:\FILE ITS\SEMESTER 7\Pra-TA\Keperluan Pra-Ta\Preliminary Pra-TA\Trying RawPy Library\buat prelim\hpmu\grey card1.dng"
    
    print("\n>>> MULAI EVALUASI <<<")
    # Cek file dulu
    if os.path.exists(file_hp_A_raw):
        process_comparison(file_hp_A_raw, file_hp_B_raw)
    else:
        print("File tidak ditemukan, silakan edit path di bagian bawah kode.")


>>> MULAI EVALUASI <<<
[LOAD] Membaca RAW: C:/Folder Dzakwan/Folder Dzakwan/Keperluan TA/Codingan Preliminary/Dataset Preliminary/hpmu/grey card1.dng
[LOAD] Membaca RAW: C:/Folder Dzakwan/Folder Dzakwan/Keperluan TA/Codingan Preliminary/Dataset Preliminary/hp najla/grey card1.dng

Mode Perbandingan: RAW vs RAW

[hp najla] pilih bagian GRAY CARD...
[hp najla] pilih bagian KONJUNGTIVA...

[hpmu] pilih bagian GRAY CARD...
[hpmu] pilih bagian KONJUNGTIVA...

[VISUALISASI] SEBELUM koreksi...

HASIL EVALUASI: RAW 1 vs RAW 2

--- 1. ANALISIS GRAY CARD ---
pembuktian kalibrasi pencahayaan berhasil.
Error Gray Card Sebelum : 89.10957 %
Error Gray Card Sesudah : 0.00089 %
   > RGB HP A (Corrected): [12944 12944 12944]
   > RGB HP B (Corrected): [12944 12944 12944]

--- 2. ANALISIS KONJUNGTIVA ---
pembuktian sbeerapa error warna konjungtiva.
Error Konjungtiva Sebelum: 105.11805 %
Error Konjungtiva Sesudah: 16.48659 %
   > RGB HP A (Corrected): [ 8287  9174 32591]
   > RGB HP B (Corrected): [1286